#### Automatic Summary Evaluation

This project uses an LLM-as-a-judge technique to evaluate summaries generated by machine learning models. Summary evaluation is useful in production environments, and has analogies in similar evaluations of GenAI outputs (or the automated evaluation of human outputs).

This notebook uses data from SummEval [1] which is a dataset of CNN and Dailymail articles, machine generated summaries of those articles, and human evaluations of those summaries. I use a question-answer generation scheme to evaluate the summaries on consistency and similar metrics.

Typically, LLM-as-a-judge means a variant of G-Eval [2] which is technique that essentially defines a metric to an LLM in the system prompt, then asks the model to rate some piece of input text using that metric. G-Eval typically has high correlation with human evaluations and is a powerful tool. It is especially powerful when the metric otherwise difficult to score - How would you the define the "relevance" of a summary using a rules-based-nlp method? How do you get thousands of "relevance" examples to train a model?

G-Eval was originally used to better measure Relevance, Consistency, Fluency, and Coherence. Relevance being the measure of how well the summary captures the key points of the article. Consistency being the measure of whether the summary reproduces all facts accurately and does not make up untrue information. Fluency being the measure of the quality of individual sentences (grammatically correct). Coherence being the measure of the quality of the summary as a whole.

In general, machine written summaries from LLMs now perform well in Fluency and Coherence - to the point that the focus can shift nearly entirely on Relevance and Consistency. The biggest struggle for LLMs is often consistency - coming from an incorrect representation of events in the source text or flat-out hallucinations. Relevance is not as "solved" as Fluency or Coherence, but LLMs often produce output with acceptable levels of relevance.

While G-Eval is a state-of-the-art metric, this notebook opts to instead use a question-answer generation (QAG) framework. QAG-based evaluation has been shown to be more correlated with human evaluation  than G-Eval for consistency and similar metrics [3]. Similar metrics referring to faithfulness (also known as groundedness) and coverage (also known as completeness), which are discussed further in the "Scoring the Summaries" section.

The idea behind QAG is to use the summaries to generate questions, then (without preserving the history that includes the summary) have the model answer those questions using the source text. This method can also be used to generate questions from the source text and to answer those questions using the summaries. The details of why to do one vs the other are also given in the "Scoring the Summaries" section.

This implementation uses [4] and [5] for inspiration but is a custom implementation and features several original ideas such question-consistency which are further discussed in the "Question - Answer Generation" section.

In [1]:
import pandas as pd
import json
import numpy as np
import aiohttp
import asyncio
import tqdm.asyncio
from aiohttp import ClientResponseError

# Dataset paths
SUMM_EVAL_DATA = "./data/summEval/model_annotations.aligned.jsonl"
TEST_FULL_TEXT = "./data/summEval/test-00000-of-00001.parquet"

# Hyperparameters for question generations
#   How many characters per question? Represents the information density of our documents
SUMMARY_CHARS_PER_Q = 60
ARTICLE_CHARS_PER_Q = 225

# LLAMA Parameters
#   Normally model parameters would be in a config file, but for simplity for a demo/Jupyter notebook I use global variables
#   With these parameters, all 1600 summaries/articles generated all questions successfully
MODEL_LLAMA = "llama3.2:3b"
URL_LLAMA = "http://localhost:11434/api/chat"
QUESTION_BUFFER_LLAMA = 2 # Number of additional questions
QUESTION_RETRIES_LLAMA = 10 # Number of times to ask extra questions
QUESTIONS_MAX_LLAMA = 10 # Max number of questions (we can get any missing on the retries)

# LLM to use
#   Can only be "llama" right now. TODO: implementation for GPT
LLM_SELECTION = "llama"

#### Reading in the Dataset

This dataset is from SummEval and comprised of full text articles from CNN and Dailymail, summaries generated by various NLP models, and human evaluations of those summaries.

TODO: move the following paragraphs to another markdown section
The goal of this notebook is to create an automated method can evaluate the faithfulness and coverage of a summary given the source article. But how will we know if we are successful?

One way is to compare the automated method's results to human evaluations on the same summary/article pair - the idea being we want to perform the task like a human would. By comparing to many such pairs, we can evaluate the method using correlation.

In [2]:
# Summaries and summary scores are available here:
#   https://storage.googleapis.com/sfr-summarization-repo-research/model_annotations.aligned.jsonl
# If you don't want to follow on a random link from me (or if that link becomes unavailable), you should be able
#   to get it through to the SummEval paper's GitHub:
#   https://github.com/Yale-LILY/SummEval?tab=readme-ov-file#human-annotations
summaries_and_labels = pd.read_json(SUMM_EVAL_DATA, lines=True)
summaries_and_labels["id"] = summaries_and_labels["id"].apply(lambda x: x.split("-")[-1])
summaries_and_labels["consistency"] = summaries_and_labels["expert_annotations"].apply(lambda x: np.mean([xi["consistency"] for xi in x]))

# The articles used in the test data of SummEval did not appear in the dataset via the download link on the SummEval
#   GitHub. I am unsure if they have been removed or if they were not intended to be made available.
# Regardless, they are HuggingFace, and confirmed that the summaries provided by SummEval match up to the articles.
#   https://huggingface.co/datasets/abisee/cnn_dailymail/tree/main/3.0.0
articles = pd.read_parquet(TEST_FULL_TEXT)

summ_eval = summaries_and_labels.merge(articles, on="id")
summ_eval.drop(["id", "filepath", "highlights", "expert_annotations", "turker_annotations", "references"], axis=1, inplace=True)
summ_eval.rename(columns={"decoded": "machine_summary"}, inplace=True)
summ_eval.to_csv("./data/summEval/summEvalTest.csv")

#### Calling the models

TODO:
* post request
* Similar payloads but different APIs, so 1 function to call llama, and another to call GPT
    * both have temperature, top_p, seed, and number of output tokens

In [3]:
# Calling Llama
#   Make sure to run "ollama serve"

async def call_llama(session, payload):
    try:
        async with session.post(URL_LLAMA, json=payload) as resp:
            response = await resp.json()
            resp.raise_for_status()

        # If the model hits the output token limit, simply return None
        if response["done_reason"] != "stop":
            return None

    except ClientResponseError as e:
        print("Llama ClientResponseError", e)
        return None
    
    except KeyError as e:
        print("Llama KeyError")
        print(resp)
        print(response)
        print(e)
        return None

    return response["message"]["content"]

def payload_llama(system_prompt, user_message, prompt_type, num_questions=None):
    return {
        "model": MODEL_LLAMA,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            },
        ],
        "format": "json",
        "stream": False,
        "options": {
            "seed": 314159265,
            "temperature": 0.0,
            "top_p": 0.0,
            "top_k": 1,
            "num_predict": 10 + 50*num_questions if prompt_type == "questions" else 10
        }, 
    }

In [4]:
# TODO: GPT's version of call_model and payload_model
# TODO: in call_gpt, on ClientResponseError implement exponential backoff (try 6 times waiting 1 second first time, then 2, then etc)
#       If error is 429, ignore it (that's hitting the quota limit) and just retry after waiting
#       If the error is anything other than 429, print the error, but still retry after waiting (likely transient network issues)

In [5]:
# Set the parameters to use based on the set parameter
if LLM_SELECTION == "llama":
    QUESTION_BUFFER = QUESTION_BUFFER_LLAMA
    QUESTION_RETRIES = QUESTION_RETRIES_LLAMA
    QUESTIONS_MAX = QUESTIONS_MAX_LLAMA
    call_model = call_llama
    model_payload = payload_llama

#### Prompt Engineering

TODO:
* json structured output (both openAI GPT and LLAMA support it) - good way to put guardrails on model output
* instructions
* one shot prompting
* system vs user message

In [6]:
# Here we define some examples for one-shot prompting
#   The summary and article examples are unrelated, but are both from the SummEval dataset
#   Neither the summary nor article are in the actual dataset used for evaluation

EXAMPLE_SUMMARY = "buckingham palace guard slipped on manhole cover in front of hundreds of horrified tourists. the queen's guard was left red-faced after he slipped on a manhole cover. he lost his footing and dropped his rifle. the incident was caught on camera. the guard is thought to have slipped because of metal shutters nailed to the soles of his boots."
EXAMPLE_SUMMARY_QUESTIONS = [
    "Did the buckingham palace guard slip on a manhole cover?",
    "Did the buckingham palace guard in front of tourists?",
    "Was the queen's guard left red-faced?",
    "Did the guard drop his rifle?",
    "Was the incident caught on camera?",
    "Is the guard thought to have slipped due to the metal shutters nailed to the soles of his boots?",
]

EXAMPLE_ARTICLE = """United Nations (CNN) -- The U.N. Security Council approved a resolution Monday to send 4,200 peacekeepers to Abyei, Sudan, as part of a recent agreement between Sudan and Southern Sudan. The resolution will establish, for six months, the United Nations Interim Security Force for Abyei (UNISFA), comprising "a maximum of 4,200 military personnel, 50 police personnel, and appropriate civilian support," the resolution states. It passed the council unanimously, 15-0. In a statement released by the State Department, Secretary Hiliary Clinton commended the swift passage of the resolution. "Abyei has been a source of regional tension for many years," the statement said. "We urge the parties to reach an immediate cease-fire and to provide aid workers with the unfettered access required to deliver humanitarian assistance to innocent civilians affected by the conflict." A week ago, the Sudanese government and the Sudan People's Liberation Movement signed an agreement to allow peacekeepers in Abyei, aimed at ending strife that has ravaged much of the country.  The two sides agreed in principle on the need for a third party to monitor the ill-defined border between north and south before the scheduled July 9 independence for the south. The U.N. peacekeepers will "monitor and verify the redeployment of any Sudan Armed Forces, Sudan People's Liberation Army or its successor" from the Abyei area, among other tasks, the Security Council resolution states."""
EXAMPLE_ARTICLE_QUESTIONS = [
   "Did the U.N. Security Council approve a resolution to send 4,200 peacekeepers to Abyei, Sudan?",
   "Was there a recent agreement between Sudan and Southern Sudan?",
   "Does the resolution establish the UNISFA?",
   "Did the resolution pass unanimously?",
   "Did Hiliary Clinton commend the swift passage of the resolution?",
   "Has Abyei been a source of regional tension for many years?",
   "Were Sudan and Southern Sudan urged to reach an immediate cease-fire?"
   "Were Sudan and Southern Sudan urged to provide aid workers with unfettered access to deliver humanitarian assistance?"
   "Did the Sudanese government sign an agreement a week ago?",
   "Did the Sudanese governemnt allow peacekeepers in Abyei?",
   "Did the two parties agree in principal for a third party to monitor the ill-defined border?",
   "Is South Sudan scheduled for independence on July 9?",
   "Will U.N. peacekeepers monitor and verify the redeployment of Sudan Armed Forces, Sudan People's Liberation Army, and its successor?",
]

In [7]:
# Define the prompts for question generation

# First, we need to define some strings to insert into our prompts
question_json_schema_as_str = """{"questions": [< question 1 >, < question 2 >]})"""
answer_json_schema_as_str = """{"answer": < answer >}"""
example_questions_as_json_str = """{"questions": [""" + ", ".join(EXAMPLE_SUMMARY_QUESTIONS) + "}]"

def question_generation_system_prompt(num_questions, example_text, example_questions):
   # We want to include a caveat with the example if the number of questions from the example differs from the number of questions we want to generate
   if len(example_questions) == num_questions:
      example_caveat = ""
   else:
      example_caveat = f"""There are {"more" if len(example_questions) > num_questions else "less"} than {num_questions} questions in the example, but the output must still have exactly {num_questions} questions. """
   
   system_prompt = f"""You are a super intelligent artificial intelligence that generates a perfectly compliant JSON with *exactly* {num_questions} close-ended questions based on text given from the user. You generate questions that can be answered with a 'yes' or a 'no', but the correct answer is always 'yes'. The schema for the JSON is {question_json_schema_as_str} where < question 1 > is the first question as a string, < question 2 > is the second question as a string, etc.
Included is an example text and example question JSON output. {example_caveat}Do *not* make questions based on the example.
Do *not* output anything except the JSON with the correct schema. Do *not* output additional information, comments, or context. Do not include newline characters. Keep the questions concise and relevant to the text.
Do *not* make up any information. Do *not* make any assumptions or inferences about the text

-- Instructions --
1) Think of a *strictly* close-ended yes-no question based on the text
2) Phrase the question so that the correct answer to the question must be 'yes'
3) Make sure this question is different from all previous questions
4) Make sure the question is related to the given text and is NOT related to the example text
5) The question should be based *only* on the given text, and should not use any external information. The question *must* be able to be answered 'yes' using *only* the given text.
6) Add the question, and only the question to the JSON
7) Repeat steps 1 - 7 until *exactly* {num_questions} close-ended questions have been generated
8) Supply the final JSON and only the JSON without any additional information, comments or context

--- Example Text ---
{example_text}

--- Example Output ---
{example_questions_as_json_str}

--- End of Example ---
"""
   
   return system_prompt


def question_generation_retry_system_prompt(num_rety_questions, existing_questions, example_text, example_questions):
   existing_questions_as_str = "\n".join(existing_questions)

   # We want to include a caveat with the example if the number of questions from the example differs from the number of questions we want to generate
   if len(example_questions) == num_rety_questions:
      example_caveat = ""
   else:
      example_caveat = f"""There are {"more" if len(example_questions) > num_rety_questions else "less"} than {num_rety_questions} questions in the example, but the output must still have exactly {num_rety_questions} questions. """
   
   system_prompt = f"""You are a super intelligent artificial intelligence that generates a perfectly compliant JSON with *exactly* {num_rety_questions} close-ended questions based on text given from the user. You generate questions that can be answered with a 'yes' or a 'no', but the correct answer is always 'yes'. The schema for the JSON is {question_json_schema_as_str} where < question 1 > is the first question as a string, < question 2 > is the second question as a string, etc.
Included is an example text and example question JSON output. {example_caveat}Do *not* make questions based on the example.
Additionally included is a set of existing questions. Your questions must be different from the existing questions.
Do *not* output anything except the JSON with the correct schema. Do *not* output additional information, comments, or context. Do not include newline characters. Keep the questions concise and relevant to the text.
Do *not* make up any information. Do *not* make any assumptions or inferences about the text

-- Instructions --
1) Think of a *strictly* close-ended yes-no question based on the text
2) Phrase the question so that the correct answer to the question must be 'yes'
3) Make sure this question is different from the existing questions and all previous questions you have given
4) Make sure the question is related to the given text and is NOT related to the example text
5) The question should be based *only* on the given text, and should not use any external information. The question *must* be able to be answered 'yes' using *only* the given text.
6) Add the question, and only the question to the JSON
7) Repeat steps 1 - 7 until *exactly* {num_rety_questions} close-ended questions have been generated
8) Supply the final JSON and only the JSON without any additional information, comments or context

--- Example Text ---
{example_text}

--- Example Output ---
{example_questions_as_json_str}

--- End of Example ---

--- Exising Questions ---
{existing_questions_as_str}

--- End of Existing Questions ---
"""
   
   return system_prompt


def question_generation_user_message(text):
   return f"""
--- Given Text---
{text}
"""

In [8]:
# Define the prompts for question answering
def answer_generation_system_prompt():
   return f"""Based on the given text and close-ended question, answer that question with a 'yes' or 'no'. The schema for the JSON is {answer_json_schema_as_str} where < answer > is a string of 'yes' or 'no'
Answers should *strictly* be 'yes' or 'no'. Do *not* provide any other answer. If the question is not close-ended and cannot be answered by a 'yes' or 'no', then make the answer 'no'
Do *not* output anything except the JSON, and the JSON should contain *only* the key "answer" with the associated asnwer string. Do *not* output additional information, comments, or context. Do *not* use newline characters. Do *not* output anything but the 'yes' or 'no' for the answer

-- Instructions --
The answer should be 'yes' or 'no' if the given text contains the information relevant to answering the question
The answer should be 'no' if the given text does not contain the information relevant to answering the question or if the answer is unknown based on the given text or if the answer is ambiguous based on the given text
If the question is not close-ended and can not be answered with a 'yes' or a 'no' then answer 'no'
"""


def answer_generation_user_message(text, question):
   return f"""
--- Given Text---
{text}

-- Question --
{question}
"""

#### Question - Answer Generation

* For question generation, we generate n close-ended yes/no questions from the question-text
    * question-text is just the text used to generate the question. For faithfulness it is the summary, for coverage it is the article
    * n depends on the length of the passage and the assumed information density. We specify different information densities for summaries/articles
    * The prompt specifies that questions be written so that the answer is always "yes". This is intended so that each question extracts 1 piece of information from the text.
        * Allowing "no" answers would allow for non-information extraction questions - In the single-shot example from the article about Sudan/South Sudan the model could ask "Did Sudan have a fireworks show for South Sudan?" to which the answer is no, but asking those kinds of questions is not useful.

* The prompt specifies that all questions are correctly answered with "yes", and we verify this by passing the question-text to the answer generation
    * Any questions that are not answered with "yes" (including open-ended responses) are removed
    * This is similar to Cycle-Consistency of cycle GANs and is used elsewhere in NLP such as translation where beginning with an English sentence, then translating to French and translating that French sentence back to English should yield the original sentence.
    * This accomplishes several useful tasks.
        * We ensure the answer to the question is "yes" and not "no". This also ensures the question is close-ended, something that Llama had some struggles with

* TODO writeup on Answer generate
* TODO: have the answer just be yes or no, don't have it return a whole JSON

In [9]:
# Define a function to generate questions/answers, and helper functions for questions/answers 

async def generate_answers(llm_call_func, llm_payload_func, session, text, questions):
    """ Generate a list of answers to the given questions using an llm

    Returns a list of answers to the given list of questions based on the given text.
    Each answer is None or a String of "yes" or "no"
    An answer is None if the model returns an invalid JSON string, does not have the key "answer" in the JSON, or the
        answer mapped to the "answer" key is not "yes" or "no"
    LLM called using answer_generation_system_prompt for the system prompt and answer_generation_user_message for the
        user message

    Parameters:
    llm_call_func - function to call LLM
    llm_payload_func - function to create payload for the LLM
    session - async aiohttp.ClientSession
    text - text to use to answer the list of questions
    questions - list of close-ended questions. Each question is a string and can be answered with 'yes' or 'no'
    """
    # Make all the answer calls in parallel
    answer_generation_tasks = [
        llm_call_func(
            session,
            llm_payload_func(
                answer_generation_system_prompt(),
                answer_generation_user_message(text, question), 
                "answer",
                )
        )
        for question in questions
    ]
    answers_as_json_strs = await asyncio.gather(*answer_generation_tasks)

    # Format the answers into a list of strings
    answers_as_list = []
    for answer_json_as_str in answers_as_json_strs:
        # By default, the answer to this question is None. Iff the response is a valid JSON string that contains the key
        # "answer" and that key that maps to a string of "yes" or "no", then the answer will be "yes" or "no"
        answer = None
        try:
            # If the response is None, that means the llm_call_func function flagged it for some reason - could be it
            #   terminated due to length rather than end of text, or a similar reason. See llm_call_func for details
            if answer_json_as_str is not None:
                answer_json = json.loads(answer_json_as_str)
                if answer_json["answer"] in ["yes", "no"]:
                    answer = answer_json["answer"]

        # If the json string was invalid or it did not contain "answer", keep the default answer
        except (json.decoder.JSONDecodeError, KeyError) as e:
            print(e)
            print("Issue with answer response\n", answer_json_as_str)

        answers_as_list.append(answer)

    return answers_as_list


async def generate_questions(llm_call_func, llm_payload_func, session, text, chars_per_q, example_text, example_questions):
    """ Generate a list of close-ended questions that can be answered with "yes" from the text using an llm
    
    Returns a list of close-ended questions (questions as strings) the given text where all answers are "yes".
    The number of questions is determined by the length of the text and chars_per_q (characters per question).
    Each question is "consistent" - generate_answers will return a list of all "yes" answers based on the input text
    LLM called using question_generation_system_prompt for the system prompt and question_generation_user_message
        for the user message

    Parameters:
    llm_call_func - function to call LLM
    llm_payload_func - function to create payload for the LLM
    session - async aiohttp.ClientSession
    text - text to base the questions on
    chars_per_q - characters per question. Assumed information density of the text
    example_text - example text to use in the LLM's system prompt
    example_questions - example questions based on example_text to use in the LLM's system prompt
    """
    # Calculate the number of questions based on the given information density of the text
    num_questions = min(QUESTIONS_MAX, np.round(len(text) / chars_per_q, 0).astype(int).item())
    
    # Generate the questions
    # Note: sometimes the model returns an incorrect number of questions (sometimes its more, sometimes less)
    #   For more, we can just truncate the list after num_questions
    #   For less, we will generate a few extra in the hopes that the few extra will cancel out the model's mistake
    #   If it's still too few questions, we have retry logic in this function
    questions_as_json_str = await llm_call_func(
        session,
        llm_payload_func(
            question_generation_system_prompt(num_questions + QUESTION_BUFFER, example_text, example_questions),
            question_generation_user_message(text),
            "questions",
            num_questions + QUESTION_BUFFER
            )
    )

    # Get the list of questions from the string of the JSON object
    try:
        # If the response is None, that means the llm_call_func function flagged it for some reason - could be it
        #   terminated due to length rather than end of text, or a similar reason. See llm_call_func for details
        if questions_as_json_str is not None:
            questions = json.loads(questions_as_json_str)["questions"]
        
        # If there was a problem, the initial call effectively returned no questions, and we will hope the retry logic
        #   does better
        else:
            questions = []

    # If there was a problem, with the json string or it did not contain "questions", set the questions to a blank list
    except (json.decoder.JSONDecodeError, KeyError) as e:
        print(e)
        print("Issue with question response\n", questions_as_json_str)
        questions = []

    # First is a consistency check - the model should answer yes to all the questions when re-given the text
    #   This is a similar to the consistency loss used in Cycle-GAN (and other domains like language translation)
    consistency_answers = await generate_answers(llm_call_func, llm_payload_func, session, text, questions)

    # Only keep the questions that are "consistent" as per the definition above
    #   Note: This also throws out answers of None (when the answer was invalid)
    questions_consistent = [
        question
        for question, answer in zip(questions, consistency_answers)
        if answer == "yes"
    ]

    # Retry QUESTION_RETRIES times, and get more aggressive each time in the number of questions generated
    for i in range(QUESTION_RETRIES):
        # Once len(questions_consistent) is greater than or equal to num_questions, stop repeating
        if len(questions_consistent) >= num_questions:
            break
        
        # Since we have too few questions, determine how many we need to generate
        #   num_questions - len(questions_consistent) would give us exactly enough
        #   We assume at least 1 will be rejected so we generate (i + 1) * QUESTION_BUFFER extra
        num_retry_questions = num_questions - len(questions_consistent) + (i + 1) * QUESTION_BUFFER
        question_retries_as_json_str = await llm_call_func(
            session,
            llm_payload_func(
                question_generation_retry_system_prompt(
                    num_retry_questions,
                    questions_consistent,
                    example_text,
                    example_questions
                ),
                question_generation_user_message(text),
                "questions",
                num_retry_questions
                )
        )

        # Same try-except as the first batch of questions
        try:
            if question_retries_as_json_str is not None:
                question_retries = json.loads(question_retries_as_json_str)["questions"]
            else:
                continue
        except (json.decoder.JSONDecodeError, KeyError) as e:
            print(e)
            print("Issue with question retry response\n", question_retries_as_json_str)
            continue

        # Same consistency check as for the first batch of questions
        consistency_retry_answers = await generate_answers(llm_call_func, llm_payload_func, session, text, question_retries)
        question_retries_consistent = [
            question
            for question, answer in zip(question_retries, consistency_retry_answers)
            if answer == "yes"
        ]
        # Concat the previous consistent questions with the new questions
        questions_consistent += question_retries_consistent

    # After all the retries, truncate down to length
    questions_consistent = questions_consistent[:num_questions]

    # If the length still isn't acceptable, inform the user but return what we have
    if len(questions_consistent) < num_questions:
        print("Issue with length of questions\n", len(questions_consistent), num_questions)

    return questions_consistent


async def generate_qnas(llm_call_func, llm_payload_func, session, question_text, answer_text, chars_per_q, example_text, example_questions):
    """Generate a list of questions from the question_text and answers to those questions using answer_text

    Returns a dataframe with a column for questions a column for answers. 
    
    Each question is a string of a close-ended yes/no question based on question_text, and the answer to the question is "yes"
    The number of questions is determined by the length of question_text and chars_per_q (characters per question)
    Each question is "consistent" - generate_answers will return a list of all "yes" answers if given question_text
    
    Each answer is associated with a question and will be 

    Parameters:
    llm_call_func - function to call LLM
    llm_payload_func - function to create payload for the LLM
    session - async aiohttp.ClientSession
    text - text to base the questions on
    chars_per_q - characters per question. Assumed information density of the text
    example_text - example text to use in the LLM's system prompt
    example_questions - example questions based on example_text to use in the LLM's system prompt
    """
    questions = await generate_questions(llm_call_func, llm_payload_func, session, question_text, chars_per_q, example_text, example_questions)
    
    # TODO: Answer the questions using the answer_text
    # answers = await generate_answers(llm_call_func, llm_payload_func, session, answer_text, questions)
    answers = ["yes"] * len(questions)

    return pd.DataFrame({"questions": questions, "answers": answers})


#### Async Processing

We have 1600 pairs of summaries and articles to process, and the longest pole for each summary/article pair is the LLM call
In order to 

In [10]:
summaries = summ_eval["machine_summary"]
articles = summ_eval["article"]
assert (len(articles) == len(summaries))

# No marginal increase in performance past 3 for Llama (on my machine)
# TODO: no expected increase in performance for GPT (due to my low rate limit)
connector = aiohttp.TCPConnector(limit=3)

# Set the timeout to 16 hours (gross overestimate)
timeout = aiohttp.ClientTimeout(total=57600) 

async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
    # Generate questions and answers to score faithfulness (aka groundedness)
    #   How much of the information in the summary is faithful to (aka grounded in) the article?
    faithfulness_tasks = [
        generate_qnas(
            call_model,
            model_payload,
            session,
            summary,
            article,
            SUMMARY_CHARS_PER_Q,
            EXAMPLE_SUMMARY,
            EXAMPLE_SUMMARY_QUESTIONS
        )
        for summary, article in zip(summaries, articles)
    ]
    print("Generating Faithfulness QnAs")
    faithfulness_qna_sets = [await task for task in tqdm.asyncio.tqdm.as_completed(faithfulness_tasks)]

    # Generate questions and answers to score coverage (aka completeness)
    #   How much of the information in the article is coverd by the summary?
    coverage_tasks = [
        generate_qnas(
            call_model,
            model_payload,
            session,
            article,
            summary,
            ARTICLE_CHARS_PER_Q,
            EXAMPLE_ARTICLE,
            EXAMPLE_ARTICLE_QUESTIONS
        )
        for summary, article in zip(summaries, articles)
    ]
    print("Generating Coverage QnAs")
    coverage_qna_sets = [await task for task in tqdm.asyncio.tqdm.as_completed(coverage_tasks)]

Generating Faithfulness QnAs


  0%|          | 0/1600 [00:00<?, ?it/s]

100%|██████████| 1600/1600 [1:43:29<00:00,  3.88s/it]   


Generating Coverage QnAs


100%|██████████| 1600/1600 [2:57:24<00:00,  6.65s/it]    


In [11]:
# GPT payload
prompt = question_generation_message(summary["machine_summaries"][0][0])
azure_gpt_payload = {
    "messages" : [{
        "role": "system",
        "content": prompt
    }],
    "temperature": 0.0,
    "top_p": 0.0,
    "seed": 1234,
    "max_token": 10 + 40*NUMBER_OF_QUESTIONS,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "output",
            "schema": {
                "type": "object",
                "properties": {
                    "question":{
                        "type": "array",
                        # "minItems": NUMBER_OF_QUESTIONS,
                        # "maxItems": NUMBER_OF_QUESTIONS,
                        "items":{
                            "type": "string",
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["questions"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
}

NameError: name 'question_generation_message' is not defined

#### Citations

[1] Fabbri, Alexander R., et al. "Summeval: Re-evaluating summarization evaluation." Transactions of the Association for Computational Linguistics 9 (2021): 391-409.

[2] Liu, Yang, et al. "G-eval: Nlg evaluation using gpt-4 with better human alignment." arXiv preprint arXiv:2303.16634 (2023).

[3] Manakul, Potsawee, Adian Liusie, and Mark JF Gales. "MQAG: Multiple-choice question answering and generation for assessing information consistency in summarization." arXiv preprint arXiv:2301.12307 (2023).

[4] Confident AI Inc. (2024). The Open-Source LLM Evaluation Framework. DeepEval. https://docs.confident-ai.com/ 

[5] TruEra. (2024). TruLens for LLMs. TruLens. https://www.trulens.org/ 